# Felix Equities — Stable Candidates (Weekend-Inclusive Scoring)

**Goal:** Find Felix equity perpetual pairs that maintain stable, positive funding even when weekend data is counted.

**Why this matters:** The existing `export_core_candidates.py` pipeline explicitly excludes weekend data
for `tradexyz`/`kinetiq` (and ignores `felix` venue entirely). This creates an optimistic view.
Positions held through weekends are affected by weekend funding — this notebook gives the conservative, honest score.

**Stability score (PRINCIPALS.md):** `0.55 × APR14 + 0.30 × APR7 + 0.15 × APR_latest`
Computed once with **all 7 days** (`stab_all`), and once **weekdays only** (`stab_wd`), to surface the weekend drag.

**Data:** `data/loris_funding_history.csv` | Venues: `tradexyz` → `xyz:`, `felix` → `flx:`, `kinetiq` → `kinetiq:`

In [1]:
# Cell 1 — Setup
import csv
import json
import statistics
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from pathlib import Path

# REPO_ROOT resolution — works in both Jupyter kernel and script context
try:
    REPO_ROOT = Path(__file__).resolve().parent.parent
except NameError:
    _nb_dir = Path(".").resolve()
    REPO_ROOT = _nb_dir if (_nb_dir / "data" / "loris_funding_history.csv").exists() else _nb_dir.parent

CSV_PATH    = REPO_ROOT / "data" / "loris_funding_history.csv"
FELIX_CACHE = REPO_ROOT / "data" / "felix_equities_cache.json"

# Venue constants
EQUITY_VENUES = {"tradexyz", "felix", "kinetiq"}
VENUE_PREFIX  = {"tradexyz": "xyz", "felix": "flx", "kinetiq": "kinetiq"}

# APR / scoring constants
ANNUALIZE        = 3.0 * 365.0 * 100.0   # 8h rate → annual %
MIN_SAMPLES_7D   = 14                      # min samples for valid 7d APR window
MIN_SAMPLES_14D  = 21                      # min samples for valid 14d APR window
FRESHNESS_WARN_H = 12.0
STABLE_FLOOR     = 10.0                    # stab_all > 10 required for STABLE tier
STABLE_POS_SHARE = 55.0                    # pos%_all > 55% required for STABLE tier
WEEKS_HIST_MIN   = 4.0                     # < 4 weeks history → LOW_HIST (score less reliable)

# Known dead pairs — confirmed delisted, cannot backfill (see docs/runbooks/loris-data-sync.md)
DEAD_PAIRS = {
    "tradexyz:AAPL", "tradexyz:META", "tradexyz:MU", "tradexyz:TSM",
    "kinetiq:AAPL",  "kinetiq:MU",
}

# ── Formatting helpers (used in all output cells) ─────────────────────────
def fmt(v, spec="+.1f"):
    return f"{v:{spec}}" if v is not None else "  N/A"

def fmt_pct(v):
    return f"{v:.0f}%" if v is not None else " N/A"

print(f"Repo root  : {REPO_ROOT}")
print(f"CSV exists : {CSV_PATH.exists()}")
print(f"Cache exists: {FELIX_CACHE.exists()}")

Repo root  : /Users/beannguyen/Development/OpenClawAgents/hip3-agent
CSV exists : True
Cache exists: True


In [2]:
# Cell 2 — Load Felix Equity Universe
with open(FELIX_CACHE) as f:
    felix_syms = set(json.load(f)["symbols"])

print(f"Felix equity universe: {len(felix_syms)} symbols")
print(f"Sample (first 10): {sorted(felix_syms)[:10]}")

Felix equity universe: 205 symbols
Sample (first 10): ['AAL', 'AAPL', 'ABBV', 'ABNB', 'ABT', 'ACHR', 'ACN', 'ADBE', 'ADI', 'AGG']


In [3]:
# Cell 3 — Load CSV (All 7 Days, No Weekend Exclusion)
#
# Key difference from core_tier_portfolio_construction.py:
#   - That script drops Sat/Sun for tradexyz/kinetiq (line 143-146)
#   - That script excludes 'felix' venue entirely (not in APPROVED_FUNDING_VENUES)
# This notebook keeps ALL rows to compute honest weekend-inclusive scores.

pairs: dict = defaultdict(list)  # (exch, sym) -> [(datetime, float)]
skipped_dead_rows = 0
loaded = 0

with open(CSV_PATH, newline="") as f:
    reader = csv.DictReader(f)
    for row in reader:
        exch, sym = row["exchange"], row["symbol"]
        if exch not in EQUITY_VENUES:
            continue
        if sym not in felix_syms:
            continue
        if f"{exch}:{sym}" in DEAD_PAIRS:
            skipped_dead_rows += 1
            continue
        try:
            ts   = datetime.fromisoformat(row["timestamp_utc"].replace("Z", "+00:00"))
            rate = float(row["funding_8h_rate"])
        except (ValueError, KeyError):
            continue
        pairs[(exch, sym)].append((ts, rate))
        loaded += 1

for k in pairs:
    pairs[k].sort(key=lambda x: x[0])

all_ts = [ts for pts in pairs.values() for ts, _ in pts]
print(f"Loaded {loaded:,} rows across {len(pairs)} equity venue pairs")
print(f"Skipped {skipped_dead_rows:,} rows from known dead pairs")
print(f"Date range: {min(all_ts).date()} → {max(all_ts).date()}")
print()
print(f"{'Pair':>20}  {'N':>5}  {'WD':>5}  {'WE':>5}  {'Last date':>12}  {'Days hist':>9}")
print("  " + "-" * 62)
for (exch, sym), pts in sorted(pairs.items(), key=lambda x: VENUE_PREFIX[x[0][0]] + ":" + x[0][1]):
    n_wd      = sum(1 for ts, _ in pts if ts.weekday() <= 4)
    n_we      = sum(1 for ts, _ in pts if ts.weekday() >= 5)
    days_hist = (pts[-1][0] - pts[0][0]).days
    prefix    = VENUE_PREFIX[exch]
    stale_mark = "  STALE" if (datetime.now(timezone.utc) - pts[-1][0]).total_seconds() / 3600 > FRESHNESS_WARN_H else ""
    print(f"{prefix}:{sym:>16}  {len(pts):>5}  {n_wd:>5}  {n_we:>5}  "
          f"{pts[-1][0].strftime('%Y-%m-%d'):>12}  {days_hist:>9}d{stale_mark}")

Loaded 29,761 rows across 28 equity venue pairs
Skipped 4,252 rows from known dead pairs
Date range: 2026-01-19 → 2026-04-27

                Pair      N     WD     WE     Last date  Days hist
  --------------------------------------------------------------
flx:            CRCL   1367    996    371    2026-04-27         97d
flx:            NVDA    977    713    264    2026-04-27         97d
flx:            TSLA   1367    996    371    2026-04-27         97d
kinetiq:            BABA   1360    989    371    2026-04-27         96d
kinetiq:            BMNR   1133    822    311    2026-04-27         58d
kinetiq:           GOOGL    636    426    210    2026-04-27         92d
kinetiq:            NVDA    855    627    228    2026-04-27         77d
kinetiq:             RTX   1109    810    299    2026-04-27         54d
kinetiq:            TSLA   1360    989    371    2026-04-27         96d
xyz:             AMD   1365    994    371    2026-04-27         97d
xyz:            AMZN    118     70    

In [4]:
# Cell 4 — Compute APR Windows Per Pair
#
# Replicates core_tier_portfolio_construction.py APR window logic
# (scripts/core_tier_portfolio_construction.py:399-408) but WITHOUT weekend exclusion.

NOW_UTC = datetime.now(timezone.utc)

def _avg_apr(series, latest_ts, days, min_samples, weekday_only=False):
    since = latest_ts - timedelta(days=days)
    if weekday_only:
        vals = [r for ts, r in series if ts >= since and ts.weekday() <= 4]
    else:
        vals = [r for ts, r in series if ts >= since]
    return (sum(vals) / len(vals) * ANNUALIZE) if len(vals) >= min_samples else None

def _pos_share(series, latest_ts, weekday_only=False, weekend_only=False):
    lookback = 30 if (latest_ts - series[0][0]).days >= 30 else 14
    since = latest_ts - timedelta(days=lookback)
    if weekday_only:
        vals = [r for ts, r in series if ts >= since and ts.weekday() <= 4]
    elif weekend_only:
        vals = [r for ts, r in series if ts >= since and ts.weekday() >= 5]
    else:
        vals = [r for ts, r in series if ts >= since]
    return (sum(1 for r in vals if r > 0) / len(vals) * 100) if vals else None

apr_data = {}

for (exch, sym), series in pairs.items():
    latest_ts   = series[-1][0]
    first_ts    = series[0][0]
    freshness_h = (NOW_UTC - latest_ts).total_seconds() / 3600
    weeks_hist  = (latest_ts - first_ts).days / 7.0
    low_hist    = weeks_hist < WEEKS_HIST_MIN   # < 4 weeks → score unreliable

    # All-7-day windows (no exclusion)
    apr_lat_all  = series[-1][1] * ANNUALIZE
    apr_7d_all   = _avg_apr(series, latest_ts, 7,  MIN_SAMPLES_7D)
    apr_14d_all  = _avg_apr(series, latest_ts, 14, MIN_SAMPLES_14D)

    # Weekday-only windows (what the existing pipeline computes)
    wd_pts       = [(ts, r) for ts, r in series if ts.weekday() <= 4]
    apr_lat_wd   = wd_pts[-1][1] * ANNUALIZE if wd_pts else None
    apr_7d_wd    = _avg_apr(series, latest_ts, 7,  MIN_SAMPLES_7D  // 2, weekday_only=True)
    apr_14d_wd   = _avg_apr(series, latest_ts, 14, MIN_SAMPLES_14D // 2, weekday_only=True)

    # Positive share
    ps_all = _pos_share(series, latest_ts)
    ps_wd  = _pos_share(series, latest_ts, weekday_only=True)
    ps_we  = _pos_share(series, latest_ts, weekend_only=True)

    apr_data[(exch, sym)] = {
        "freshness_h":  freshness_h,
        "weeks_hist":   weeks_hist,
        "low_hist":     low_hist,
        "n":            len(series),
        "apr_lat_all":  apr_lat_all,
        "apr_7d_all":   apr_7d_all,
        "apr_14d_all":  apr_14d_all,
        "apr_lat_wd":   apr_lat_wd,
        "apr_7d_wd":    apr_7d_wd,
        "apr_14d_wd":   apr_14d_wd,
        "ps_all":       ps_all,
        "ps_wd":        ps_wd,
        "ps_we":        ps_we,
    }

n_fresh    = sum(1 for d in apr_data.values() if d["freshness_h"] <= FRESHNESS_WARN_H)
n_low_hist = sum(1 for d in apr_data.values() if d["low_hist"])
print(f"APR windows computed for {len(apr_data)} pairs")
print(f"Fresh (≤{FRESHNESS_WARN_H:.0f}h): {n_fresh}  |  Stale: {len(apr_data) - n_fresh}")
print(f"LOW_HIST (< {WEEKS_HIST_MIN:.0f} weeks data): {n_low_hist} — scores less reliable for these")

APR windows computed for 28 pairs
Fresh (≤12h): 28  |  Stale: 0
LOW_HIST (< 4 weeks data): 4 — scores less reliable for these


In [5]:
# Cell 5 — Compute Weekend-Inclusive Stability Score + Tier Classification
#
# Formula (PRINCIPALS.md §3): stability = 0.55×APR14 + 0.30×APR7 + 0.15×APR_latest
# stab_all = computed from all-7-day data  (conservative / honest)
# stab_wd  = computed from weekday-only data (optimistic, matches export_core_candidates.py)
# delta    = stab_all − stab_wd  (negative = weekends drag score down)

def _stability(apr14, apr7, apr_lat):
    if any(v is None for v in [apr14, apr7, apr_lat]):
        return None
    return 0.55 * apr14 + 0.30 * apr7 + 0.15 * apr_lat

def _flip_risk(ps_wd, ps_we):
    return (ps_we is not None and ps_wd is not None
            and ps_we < 50.0 and ps_wd >= 60.0)

def _tier(stab_all, ps_all, flip_risk):
    if stab_all is None:
        return "DATA?"
    if flip_risk or stab_all <= 0.0:
        return "AVOID"
    if stab_all >= STABLE_FLOOR and ps_all is not None and ps_all >= STABLE_POS_SHARE:
        return "STABLE"
    return "MONITOR"

scored = []

for (exch, sym), d in apr_data.items():
    stab_all = _stability(d["apr_14d_all"], d["apr_7d_all"], d["apr_lat_all"])
    stab_wd  = _stability(d["apr_14d_wd"],  d["apr_7d_wd"],  d["apr_lat_wd"])
    delta    = (stab_all - stab_wd) if (stab_all is not None and stab_wd is not None) else None
    flip     = _flip_risk(d["ps_wd"], d["ps_we"])
    tier     = _tier(stab_all, d["ps_all"], flip)

    scored.append({
        "pair":        f"{VENUE_PREFIX[exch]}:{sym}",
        "exch":        exch,
        "sym":         sym,
        "tier":        tier,
        "stab_all":    stab_all,
        "stab_wd":     stab_wd,
        "delta":       delta,
        "apr_14d_all": d["apr_14d_all"],
        "apr_7d_all":  d["apr_7d_all"],
        "apr_lat_all": d["apr_lat_all"],
        "apr_14d_wd":  d["apr_14d_wd"],
        "ps_all":      d["ps_all"],
        "ps_wd":       d["ps_wd"],
        "ps_we":       d["ps_we"],
        "freshness_h": d["freshness_h"],
        "weeks_hist":  d["weeks_hist"],
        "low_hist":    d["low_hist"],
        "flip_risk":   flip,
    })

scored.sort(key=lambda r: r["stab_all"] if r["stab_all"] is not None else float("-inf"), reverse=True)

n_stable   = sum(1 for r in scored if r["tier"] == "STABLE")
n_monitor  = sum(1 for r in scored if r["tier"] == "MONITOR")
n_avoid    = sum(1 for r in scored if r["tier"] == "AVOID")
n_flip     = sum(1 for r in scored if r["flip_risk"])
n_low_hist = sum(1 for r in scored if r["low_hist"])

print(f"Stability scores computed for {len(scored)} pairs")
print()
print(f"  STABLE  (stab_all ≥ {STABLE_FLOOR}, pos%_all ≥ {STABLE_POS_SHARE}%): {n_stable}")
print(f"  MONITOR (positive but weekend-degraded):                         {n_monitor}")
print(f"  AVOID   (stab_all ≤ 0 or flip risk):                             {n_avoid}")
print(f"  Flip risk:                                                        {n_flip}")
print(f"  LOW_HIST (< {WEEKS_HIST_MIN:.0f} wks data, score uncertain):              {n_low_hist}")

Stability scores computed for 28 pairs

  STABLE  (stab_all ≥ 10.0, pos%_all ≥ 55.0%): 9
  MONITOR (positive but weekend-degraded):                         11
  AVOID   (stab_all ≤ 0 or flip risk):                             8
  Flip risk:                                                        6
  LOW_HIST (< 4 wks data, score uncertain):              4


In [6]:
# Cell 6 — Main Results Table (sorted by stab_all descending)
#
# wks_hist = weeks of historical data for this pair
# LOW_HIST  = flag when < 4 weeks — stability score is based on limited observations;
#             treat these pairs as MONITOR regardless of tier label until more data accumulates

print("Felix Equities — Weekend-Inclusive Stability Ranking")
print("Generated:", datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC"))
print()
print("  stab_all = stability score with ALL 7 days     ← honest/conservative")
print("  stab_wd  = stability score weekdays ONLY       ← what export_core_candidates shows")
print("  delta    = stab_all − stab_wd  (negative = weekends drag score down)")
print("  wks      = weeks of historical data  [LOW_HIST < 4 wks → score uncertain]")
print()

hdr = (
    f"{'Pair':>20}  {'Tier':>6}  "
    f"{'stab_all':>9}  {'stab_wd':>8}  {'delta':>7}  "
    f"{'APR14':>7}  {'APR7':>7}  {'APR_lat':>8}  "
    f"{'pos%all':>7}  {'pos%wd':>6}  {'pos%we':>6}  "
    f"{'Flip':>4}  {'wks':>5}  {'flags':>8}"
)
print(hdr)
print("  " + "-" * (len(hdr) - 2))

for r in scored:
    tier_disp  = r["tier"]
    flip_disp  = "Y" if r["flip_risk"] else ""
    stale_mark = "!" if r["freshness_h"] > FRESHNESS_WARN_H else " "
    flags      = ("LOW_HIST" if r["low_hist"] else "")
    print(
        f"{r['pair']:>20}  {tier_disp:>6}  "
        f"{fmt(r['stab_all']):>9}  {fmt(r['stab_wd']):>8}  {fmt(r['delta']):>7}  "
        f"{fmt(r['apr_14d_all']):>7}  {fmt(r['apr_7d_all']):>7}  {fmt(r['apr_lat_all']):>8}  "
        f"{fmt_pct(r['ps_all']):>7}  {fmt_pct(r['ps_wd']):>6}  {fmt_pct(r['ps_we']):>6}  "
        f"{flip_disp:>4}  {r['weeks_hist']:>4.1f}w  {flags:<8}"
    )

Felix Equities — Weekend-Inclusive Stability Ranking
Generated: 2026-04-27 02:53 UTC

  stab_all = stability score with ALL 7 days     ← honest/conservative
  stab_wd  = stability score weekdays ONLY       ← what export_core_candidates shows
  delta    = stab_all − stab_wd  (negative = weekends drag score down)
  wks      = weeks of historical data  [LOW_HIST < 4 wks → score uncertain]

                Pair    Tier   stab_all   stab_wd    delta    APR14     APR7   APR_lat  pos%all  pos%wd  pos%we  Flip    wks     flags
  ------------------------------------------------------------------------------------------------------------------------------------
            xyz:HIMS  STABLE      +76.9     +89.6    -12.7   +110.8    +61.1     -15.8      84%     82%     91%         5.6w          
             xyz:LLY  STABLE      +60.5     +49.6    +10.9    +28.5    +51.2    +196.6      76%     84%     54%        13.9w          
            xyz:RIVN  STABLE      +53.7     +49.2     +4.5    +43.7   

In [7]:
# Cell 7 — Tier Summary Tables

def _prep(r):
    return {
        "pair":     r["pair"],
        "stab_all": fmt(r["stab_all"]),
        "stab_wd":  fmt(r["stab_wd"]),
        "delta":    fmt(r["delta"]),
        "apr14":    fmt(r["apr_14d_all"]),
        "apr7":     fmt(r["apr_7d_all"]),
        "ps_all":   fmt_pct(r["ps_all"]),
        "ps_wd":    fmt_pct(r["ps_wd"]),
        "ps_we":    fmt_pct(r["ps_we"]),
        "flip":     "FLIP" if r["flip_risk"] else "",
        "wks":      f"{r['weeks_hist']:.1f}w",
        "flags":    "LOW_HIST" if r["low_hist"] else "",
    }

def _print_tier(title, rows, cols):
    print(f"{'='*72}")
    print(f"  {title}")
    print(f"{'='*72}")
    if not rows:
        print("  (none)")
        print()
        return
    hdr = "  " + "  ".join(f"{c[0]:>{c[1]}}" for c in cols)
    print(hdr)
    print("  " + "-" * (sum(c[1] for c in cols) + 2 * len(cols)))
    for r in rows:
        line = "  " + "  ".join(f"{str(r.get(c[2], '')):>{c[1]}}" for c in cols)
        print(line)
    print()

stable_rows  = [_prep(r) for r in scored if r["tier"] == "STABLE"]
monitor_rows = [_prep(r) for r in scored if r["tier"] == "MONITOR"]
avoid_rows   = [_prep(r) for r in scored if r["tier"] == "AVOID"]

COLS_STABLE = [
    ("pair",20,"pair"),("stab_all",9,"stab_all"),("stab_wd",8,"stab_wd"),("delta",7,"delta"),
    ("apr14",7,"apr14"),("apr7",7,"apr7"),("ps_all",7,"ps_all"),("ps_we",6,"ps_we"),
    ("wks",6,"wks"),("flags",8,"flags"),
]
COLS_MONITOR = [
    ("pair",20,"pair"),("stab_all",9,"stab_all"),("stab_wd",8,"stab_wd"),("delta",7,"delta"),
    ("apr14",7,"apr14"),("ps_wd",6,"ps_wd"),("ps_we",6,"ps_we"),("flip",4,"flip"),
    ("wks",6,"wks"),("flags",8,"flags"),
]
COLS_AVOID = [
    ("pair",20,"pair"),("stab_all",9,"stab_all"),("stab_wd",8,"stab_wd"),("delta",7,"delta"),
    ("apr14",7,"apr14"),("ps_wd",6,"ps_wd"),("ps_we",6,"ps_we"),("flip",4,"flip"),
    ("wks",6,"wks"),
]

_print_tier(
    f"STABLE — safe to hold through weekends ({len(stable_rows)} pairs, stab_all ≥ {STABLE_FLOOR}, pos%_all ≥ {STABLE_POS_SHARE}%)",
    stable_rows, COLS_STABLE,
)

_print_tier(
    f"MONITOR — positive but weekend-degraded ({len(monitor_rows)} pairs)",
    monitor_rows, COLS_MONITOR,
)

_print_tier(
    f"AVOID — negative weekend-inclusive score or flip risk ({len(avoid_rows)} pairs)",
    avoid_rows, COLS_AVOID,
)

# LOW_HIST callout
low_hist_stable = [r for r in scored if r["tier"] == "STABLE" and r["low_hist"]]
if low_hist_stable:
    print(f"⚠  LOW_HIST caution — these are classified STABLE but have < {WEEKS_HIST_MIN:.0f} weeks of data:")
    for r in low_hist_stable:
        n_we = sum(1 for ts, _ in pairs[(r["exch"], r["sym"])] if ts.weekday() >= 5)
        print(f"   {r['pair']:>20}: {r['weeks_hist']:.1f} wks history, {n_we} weekend observations — "
              f"treat as MONITOR until ≥ {WEEKS_HIST_MIN:.0f} wks confirmed")
    print()

# Venue breakdown
print("Venue breakdown:")
for venue in ["tradexyz", "felix", "kinetiq"]:
    prefix = VENUE_PREFIX[venue]
    v_rows = [r for r in scored if r["exch"] == venue]
    t = {k: sum(1 for r in v_rows if r["tier"] == k) for k in ("STABLE","MONITOR","AVOID","DATA?")}
    lh = sum(1 for r in v_rows if r["low_hist"])
    print(f"  {prefix:>8}: {len(v_rows)} pairs — STABLE={t['STABLE']}, MONITOR={t['MONITOR']}, AVOID={t['AVOID']}  (LOW_HIST={lh})")

  STABLE — safe to hold through weekends (9 pairs, stab_all ≥ 10.0, pos%_all ≥ 55.0%)
                  pair   stab_all   stab_wd    delta    apr14     apr7   ps_all   ps_we     wks     flags
  ---------------------------------------------------------------------------------------------------------
              xyz:HIMS      +76.9     +89.6    -12.7   +110.8    +61.1      84%     91%    5.6w          
               xyz:LLY      +60.5     +49.6    +10.9    +28.5    +51.2      76%     54%   13.9w          
              xyz:RIVN      +53.7     +49.2     +4.5    +43.7    +52.0      80%     55%   13.9w          
              xyz:COST      +43.5     +34.1     +9.4    +48.4    +46.3      79%     85%   13.9w          
              xyz:ORCL      +18.3     +21.4     -3.1    +17.4    +10.0      79%     51%   13.9w          
               xyz:AMD      +15.2     +13.6     +1.6    +12.9    +16.1      85%     54%   13.9w          
          kinetiq:BMNR      +13.9      -3.4    +17.3    +12.2   

In [8]:
# Cell 8 — Key Takeaway & Trading Implications

print("=" * 72)
print("  TRADING IMPLICATIONS")
print("=" * 72)
print()

# Pipeline optimism vs reality
stab_all_vals = [r["stab_all"] for r in scored if r["stab_all"] is not None]
stab_wd_vals  = [r["stab_wd"]  for r in scored if r["stab_wd"]  is not None]
delta_vals    = [r["delta"]    for r in scored if r["delta"]    is not None]

if stab_all_vals and stab_wd_vals:
    avg_all = sum(stab_all_vals) / len(stab_all_vals)
    avg_wd  = sum(stab_wd_vals)  / len(stab_wd_vals)
    avg_d   = sum(delta_vals)    / len(delta_vals) if delta_vals else None
    n_wd_higher = sum(1 for r in scored
                      if r["stab_wd"] is not None and r["stab_all"] is not None
                      and r["stab_wd"] > r["stab_all"])
    print(f"Pipeline view (weekdays only):    avg stab_wd  = {avg_wd:>+.1f}")
    print(f"Conservative view (all 7 days):   avg stab_all = {avg_all:>+.1f}")
    if avg_d is not None:
        print(f"Weekend drag (avg delta):         {avg_d:>+.1f}  (negative = pipeline is optimistic)")
    print(f"Pairs where stab_wd > stab_all:   {n_wd_higher}/{len(scored)} ({100*n_wd_higher//len(scored)}%)")
    print()

# STABLE tier — actionable (filter out LOW_HIST for high-confidence list)
stable_hi_conf = [r for r in scored if r["tier"] == "STABLE" and not r["low_hist"]]
stable_low_conf = [r for r in scored if r["tier"] == "STABLE" and r["low_hist"]]

if stable_hi_conf:
    print(f"HOLD THROUGH WEEKEND — {len(stable_hi_conf)} high-confidence STABLE pairs:")
    for r in stable_hi_conf:
        print(f"  {r['pair']:>20}: stab_all={fmt(r['stab_all'])}, APR14={fmt(r['apr_14d_all'])}, "
              f"pos%we={fmt_pct(r['ps_we'])}, delta={fmt(r['delta'])}  [{r['weeks_hist']:.0f} wks]")
    print()

if stable_low_conf:
    print(f"TENTATIVE STABLE — {len(stable_low_conf)} pairs (LOW_HIST: < {WEEKS_HIST_MIN:.0f} wks, treat as MONITOR):")
    for r in stable_low_conf:
        print(f"  {r['pair']:>20}: stab_all={fmt(r['stab_all'])}, {r['weeks_hist']:.1f} wks data — "
              f"wait for more weekend observations before committing")
    print()

# FLIP RISK — must act
flip_pairs = [r for r in scored if r["flip_risk"]]
if flip_pairs:
    print(f"CLOSE FRIDAY EOD — {len(flip_pairs)} flip-risk pairs (weekend pos% < 50%):")
    for r in sorted(flip_pairs, key=lambda x: (x["stab_wd"] or 0), reverse=True):
        print(f"  {r['pair']:>20}: stab_wd={fmt(r['stab_wd'])} → stab_all={fmt(r['stab_all'])}, "
              f"pos%wd={fmt_pct(r['ps_wd'])} → pos%we={fmt_pct(r['ps_we'])}  [{r['weeks_hist']:.0f} wks]")
    print()

# MONITOR — decision guidance
monitor = [r for r in scored if r["tier"] == "MONITOR"]
if monitor:
    print(f"MONITOR — {len(monitor)} pairs: positive but weekend-degraded.")
    print("  Rule: if stab_all > 5.0 AND pos%we > 50% → hold OK.")
    print("        otherwise → close Friday, reopen Monday.")
    print()
    for r in monitor:
        hold_ok = (r["stab_all"] is not None and r["stab_all"] > 5.0
                   and r["ps_we"] is not None and r["ps_we"] > 50.0)
        rec = "hold OK" if hold_ok else "close Fri"
        lh  = " [LOW_HIST]" if r["low_hist"] else ""
        print(f"  {r['pair']:>20}: stab_all={fmt(r['stab_all'])}  delta={fmt(r['delta'])}  "
              f"pos%we={fmt_pct(r['ps_we'])}  [{r['weeks_hist']:.0f}w]  → {rec}{lh}")
    print()

# Felix (flx:) venue note
flx = [r for r in scored if r["exch"] == "felix"]
if flx:
    print("FELIX (flx:) VENUE NOTE:")
    print("  Felix equity funding zeroes out on weekends (pos%we ≈ 1-5%).")
    print("  No funding LOSS — funding simply stops. Capital is idle 2/7 days (~28%).")
    print("  Adjust effective APR: APR14_all × (5/7) = 5-day equivalent annual rate.")
    print()
    for r in flx:
        adj = r["apr_14d_all"] * (5.0 / 7.0) if r["apr_14d_all"] is not None else None
        print(f"  {r['pair']:>20}: APR14={fmt(r['apr_14d_all'])}  5/7-adj={fmt(adj)}  "
              f"pos%we={fmt_pct(r['ps_we'])}  [{r['weeks_hist']:.0f}w]")
    print()

print("ROUND-TRIP COST ESTIMATE (Friday close + Monday reopen, 1 leg each):")
print("  ~0.07% of notional (0.035% taker × perp leg)")
print("  $20k position: ~$14/cycle  |  $50k: ~$35/cycle")
print()
print("DECISION RULE:")
print("  STABLE (high-conf) → do NOT close Friday (weekend drag < round-trip cost)")
print("  STABLE (LOW_HIST)  → treat as MONITOR until ≥ 4 wks of weekends confirmed")
print("  FLIP               → ALWAYS close Friday (weekend is net negative)")
print("  MONITOR            → compute expected 2-day loss vs $14-$35, close if loss > cost")

  TRADING IMPLICATIONS

Pipeline view (weekdays only):    avg stab_wd  = +15.2
Conservative view (all 7 days):   avg stab_all = +13.7
Weekend drag (avg delta):         -1.5  (negative = pipeline is optimistic)
Pairs where stab_wd > stab_all:   18/28 (64%)

HOLD THROUGH WEEKEND — 8 high-confidence STABLE pairs:
              xyz:HIMS: stab_all=+76.9, APR14=+110.8, pos%we=91%, delta=-12.7  [6 wks]
               xyz:LLY: stab_all=+60.5, APR14=+28.5, pos%we=54%, delta=+10.9  [14 wks]
              xyz:RIVN: stab_all=+53.7, APR14=+43.7, pos%we=55%, delta=+4.5  [14 wks]
              xyz:COST: stab_all=+43.5, APR14=+48.4, pos%we=85%, delta=+9.4  [14 wks]
              xyz:ORCL: stab_all=+18.3, APR14=+17.4, pos%we=51%, delta=-3.1  [14 wks]
               xyz:AMD: stab_all=+15.2, APR14=+12.9, pos%we=54%, delta=+1.6  [14 wks]
          kinetiq:BMNR: stab_all=+13.9, APR14=+12.2, pos%we=56%, delta=+17.3  [8 wks]
              xyz:MSFT: stab_all=+10.2, APR14=+8.6, pos%we=64%, delta=-1.1  [14 wks]